# SPANDAN · 01 · Fault diagnosis on CWRU (tasks M2-19, M2-22, M2-23)

Trains the bearing-fault classifier on the **CWRU** drive-end data, resampled to our sensor's 3.2 kHz,
and evaluates it without leakage (leave-one-load-out and grouped by recording).

**Before running**
1. *Add data*: attach a CWRU `.mat` mirror, e.g. `sufian79/cwru-mat-full-dataset` or `esraakhaled299/cwru-data`.
2. *Settings*: turn **Internet on** (to clone the repo). CPU is enough; no GPU needed.
3. *Run all*. Takes a few minutes. The first cell installs the scikit-learn version pinned in
   `ml/requirements.txt` (the backend uses the same one, so the model loads there).

**After running**: download `/kaggle/working/results/` (Output tab) and copy it into `ml/results/` in the repo:
`diagnosis-hgb-v1.joblib`, `diagnosis-hgb-v1_report.json`, `cwru_manifest.csv`, `confusion_*.png`.

In [ ]:
# Get the SPANDAN code (public repo). Needs "Internet on" in the notebook settings.
import os, re, sys, subprocess
REPO = "/kaggle/working/SPANDAN"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)  # rerun in the same session: get the latest code
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Duggineniakhil/SPANDAN.git", REPO], check=True)
sys.path.insert(0, f"{REPO}/ml")

# Install the exact scikit-learn version the backend uses, so the saved model loads there.
# Must run before anything imports sklearn (if it was imported already: Run > Restart & clear outputs, then Run all).
pin = re.search(r"^scikit-learn==([\d.]+)", open(f"{REPO}/ml/requirements.txt").read(), re.M).group(1)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={pin}"], check=True)
import numpy, scipy, sklearn
print("numpy", numpy.__version__, "| scipy", scipy.__version__, "| scikit-learn", sklearn.__version__)
assert sklearn.__version__ == pin, f"scikit-learn {sklearn.__version__} is loaded, need {pin}: restart the kernel and run all"


In [ ]:
# Find the attached CWRU files
from pathlib import Path
mats = list(Path("/kaggle/input").rglob("*.mat"))
print(len(mats), "mat files, e.g.", [m.name for m in mats[:8]])
assert mats, "No .mat files: attach a CWRU dataset (Add data)"

In [ ]:
from spandan_ml.pipelines import run_cwru
report = run_cwru("/kaggle/input", "/kaggle/working/results", window_s=2.0, hop_s=1.0, max_windows=40)

## Check the labels
Every row of the manifest is one recording (= one split unit). Check that labels, loads and sampling
rates look right before trusting the numbers: `label_source` says whether the label came from the file
name or the CWRU number inside the file, and the normal files' `fs` was measured from the signal.

In [ ]:
import csv
rows = list(csv.DictReader(open("/kaggle/working/results/cwru_manifest.csv")))
from collections import Counter
print(Counter((r["label"], r["fs"]) for r in rows))
for r in rows[:12]: print(r)

In [ ]:
import json
from IPython.display import Image, display
for k in ("leave_one_load_out", "grouped_5fold_by_recording", "rules_only"):
    r = report[k]
    print(f"{k:28s} accuracy {r['accuracy']:.3f}  macro-F1 {r['macro_f1']:.3f}  n={r['n']}",
          f"unknown {r.get('unknown_rate')}" if 'unknown_rate' in r else "")
    display(Image(f"/kaggle/working/results/confusion_{k}.png"))